In [5]:
import os
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [6]:
files = [
    'two_class_1s_no.csv',
    'two_class_1s_0.5.csv',
    'two_class_1s_0.8.csv',
    'two_class_2s_no.csv',
    'two_class_2s_0.5.csv',
    'two_class_2s_0.8.csv',
    'two_class_3s_no.csv',
    'two_class_3s_0.5.csv',
    'two_class_3s_0.8.csv',
    'two_class_4s_no.csv',
    'two_class_4s_0.5.csv',
    'two_class_4s_0.8.csv',
    'two_class_5s_no.csv',
    'two_class_5s_0.5.csv',
    'two_class_5s_0.8.csv'
]

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/extracted_features_data/3_class/train_set'

In [7]:
def screening_cross_val(X, y, groups, model_type, random_state=42):
    """
    Compare different sampling techniques for imbalanced classification
    """
    
    results = {}
    conf_mat_data = {}
    
        
    # Cross-validation setup
    # gkf = GroupKFold(n_splits=5)  # Using 5 folds for faster initial testing
    sgkf = StratifiedGroupKFold(n_splits=5)  # Stratified Group K-Fold for better class balance
    fold_results = []
        
    # For confusion matrix
    predicted_targets = np.array([])
    actual_targets = np.array([])
        
    # split data
    for fold_num, (train_idx_fold, test_idx_fold) in enumerate(sgkf.split(X, y, groups)):
        X_train_fold, X_test_fold = X.iloc[train_idx_fold], X.iloc[test_idx_fold]
        y_train_fold, y_test_fold = y.iloc[train_idx_fold], y.iloc[test_idx_fold]

            
        if model_type == 'xgb':
            # Use XGBoost with DEFAULT parameters
            model = XGBClassifier(
                random_state=random_state,
                eval_metric='logloss',  # Suppress warning
            )
        elif model_type == 'dt':
            # Use Decision Tree with DEFAULT parameters
            model = DecisionTreeClassifier(
                random_state=random_state,
            )
        elif model_type == 'rf':
            # Use RandomForest with DEFAULT parameters
            model = RandomForestClassifier(
                random_state=random_state,
            )
            
        # Encode labels
        label_encoder = LabelEncoder()
        y_train_encoded = label_encoder.fit_transform(y_train_fold)
            
        # Fit model
        model.fit(X_train_fold, y_train_encoded)
            
        # Predict
        predictions = label_encoder.inverse_transform(model.predict(X_test_fold))
            
        # For confusion matrix
        predicted_targets = np.append(predicted_targets, predictions)
        actual_targets = np.append(actual_targets, y_test_fold)
        # Store the confusion matrix data
        conf_mat_data = {
            'predicted': predicted_targets,
            'actual': actual_targets
        }
            
        # Calculate met# split datarics
        accuracy = accuracy_score(y_test_fold, predictions)
        report_dict = classification_report(y_test_fold, predictions, output_dict=True)
        
        precision_pre_void = report_dict.get("pre-void", {}).get("precision", 0.0)
        precision_void = report_dict.get("void", {}).get("precision", 0.0)
        precision_post_void = report_dict.get("post-void", {}).get("precision", 0.0)
        
        recall_pre_void = report_dict.get("pre-void", {}).get("recall", 0.0)
        recall_void = report_dict.get("void", {}).get("recall", 0.0)
        recall_post_void = report_dict.get("post-void", {}).get("recall", 0.0)
        
        f1_pre_void = report_dict.get("pre-void", {}).get("f1-score", 0.0)
        f1_void = report_dict.get("void", {}).get("f1-score", 0.0)
        f1_post_void = report_dict.get("post-void", {}).get("f1-score", 0.0)
        
        macro_f1 = report_dict.get("macro avg", {}).get("f1-score", 0.0)
        macro_precision = report_dict.get("macro avg", {}).get("precision", 0.0)
        macro_recall = report_dict.get("macro avg", {}).get("recall", 0.0)
        
        weighted_f1 = report_dict.get("weighted avg", {}).get("f1-score", 0.0)
        weighted_precision = report_dict.get("weighted avg", {}).get("precision", 0.0)
        weighted_recall = report_dict.get("weighted avg", {}).get("recall", 0.0)

            
        # Store detailed results
        fold_results.append({
            'fold': fold_num,
            'accuracy': accuracy,
            'f1_pre_void': f1_pre_void,
            'f1_void': f1_void,
            'f1_post_void': f1_post_void,
            'precision_pre_void': precision_pre_void,
            'precision_void': precision_void,
            'precision_post_void': precision_post_void,
            'recall_pre_void': recall_pre_void,
            'recall_void': recall_void,
            'recall_post_void': recall_post_void,
            'macro_f1': macro_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'weighted_f1': weighted_f1,
            'weighted_precision': weighted_precision,
            'weighted_recall': weighted_recall,
            'class_distribution_train': dict(y_train_fold.value_counts()),
            'class_distribution_test': dict(y_test_fold.value_counts())
        })
            

        # Calculate summary statistics
        if fold_results:  # Only if we have valid results
            results = {
                'mean_accuracy': np.mean([f['accuracy'] for f in fold_results]),
                'std_accuracy': np.std([f['accuracy'] for f in fold_results]),
                
                'mean_macro_precision': np.mean([f['macro_precision'] for f in fold_results]),
                'std_macro_precision': np.std([f['macro_precision'] for f in fold_results]),
                'mean_macro_recall': np.mean([f['macro_recall'] for f in fold_results]),
                'std_macro_recall': np.std([f['macro_recall'] for f in fold_results]),
                'mean_macro_f1': np.mean([f['macro_f1'] for f in fold_results]),
                'std_macro_f1': np.std([f['macro_f1'] for f in fold_results]),
                
                
                'mean_weighted_precision': np.mean([f['weighted_precision'] for f in fold_results]),
                'std_weighted_precision': np.std([f['weighted_precision'] for f in  fold_results]),
                'mean_weighted_recall': np.mean([f['weighted_recall'] for f in fold_results]),
                'std_weighted_recall': np.std([f['weighted_recall'] for f in fold_results]),
                'mean_weighted_f1': np.mean([f['weighted_f1'] for f in fold_results]),
                'std_weighted_f1': np.std([f['weighted_f1'] for f in fold_results]),
                
                'mean_f1_pre_void': np.mean([f['f1_pre_void'] for f in fold_results]),
                'std_f1_pre_void': np.std([f['f1_pre_void'] for f in fold_results]),
                'mean_f1_void': np.mean([f['f1_void'] for f in fold_results]),
                'std_f1_void': np.std([f['f1_void'] for f in fold_results]),
                'mean_f1_post_void': np.mean([f['f1_post_void'] for f in fold_results]),
                'std_f1_post_void': np.std([f['f1_post_void'] for f in fold_results]),   
                
                'mean_precision_pre_void': np.mean([f['precision_pre_void'] for f in fold_results]),
                'std_precision_pre_void': np.std([f['precision_pre_void'] for f in fold_results]),
                'mean_precision_void': np.mean([f['precision_void'] for f in fold_results]),
                'std_precision_void': np.std([f['precision_void'] for f in fold_results]),
                'mean_precision_post_void': np.mean([f['precision_post_void'] for f in fold_results]),
                'std_precision_post_void': np.std([f['precision_post_void'] for f in fold_results]),
                
                'mean_recall_pre_void': np.mean([f['recall_pre_void'] for f in fold_results]),
                'std_recall_pre_void': np.std([f['recall_pre_void'] for f in fold_results]),
                'mean_recall_void': np.mean([f['recall_void'] for f in fold_results]),
                'std_recall_void': np.std([f['recall_void'] for f in fold_results]),
                'mean_recall_post_void': np.mean([f['recall_post_void'] for f in fold_results]),
                'std_recall_post_void': np.std([f['recall_post_void'] for f in fold_results]),
                
                

                'fold_details': fold_results
            }
    
    return results, conf_mat_data

## Decision Tree

In [8]:
file_results_dt = {}
conf_mat_data = {}

for file in tqdm(files, desc="Producing results for different sampling techniques - train data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_dt[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'dt', 42)
    
# pickle the reults
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_dt_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_dt, f)
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_dt_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)   

Producing results for different sampling techniques - train data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no


Producing results for different sampling techniques - train data only:   7%|▋         | 1/15 [00:01<00:15,  1.13s/it]

Analysing 1s_0.5


Producing results for different sampling techniques - train data only:  13%|█▎        | 2/15 [00:03<00:23,  1.79s/it]

...

Producing results for different sampling techniques - train data only:  87%|████████▋ | 13/15 [00:17<00:01,  1.42it/s]

Analysing 5s_no
Analysing 5s_0.5


Producing results for different sampling techniques - train data only:  93%|█████████▎| 14/15 [00:18<00:00,  1.68it/s]

Analysing 5s_0.8


Producing results for different sampling techniques - train data only: 100%|██████████| 15/15 [00:19<00:00,  1.28s/it]


## Random Forest

In [9]:
file_results_rf = {}
conf_mat_data = {}
for file in tqdm(files, desc="Producing results for different sampling techniques - working data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_rf[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'rf', 42)
    
# pickle the results
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_rf_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_rf, f)
    
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_rf_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)

Producing results for different sampling techniques - working data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no


Producing results for different sampling techniques - working data only:   7%|▋         | 1/15 [00:05<01:12,  5.17s/it]

Analysing 1s_0.5


Producing results for different sampling techniques - working data only:  13%|█▎        | 2/15 [00:16<01:51,  8.56s/it]

...

Producing results for different sampling techniques - working data only:  87%|████████▋ | 13/15 [01:29<00:07,  3.62s/it]

Analysing 5s_0.5


Producing results for different sampling techniques - working data only:  93%|█████████▎| 14/15 [01:31<00:03,  3.07s/it]

Analysing 5s_0.8


Producing results for different sampling techniques - working data only: 100%|██████████| 15/15 [01:35<00:00,  6.39s/it]


## XGBoost

In [10]:
file_results_xgb = {}
conf_mat_data = {}
for file in tqdm(files, desc="Producing results for different sampling techniques - working data only"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    details = file.split('_')
    exp_name = f"{details[2]}_{details[-1].replace('.csv', '')}" 
    print(f"Analysing {exp_name}")
    
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    file_results_xgb[exp_name], conf_mat_data[exp_name] = screening_cross_val(X, y, groups,'xgb', 42)
    
# pickle the results
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_xgb_cv_results_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(file_results_xgb, f)
# pickle the confusion matrix data
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/training_results/3_class_xgb_conf_mat_data_5_fold_stratified.pkl', 'wb') as f:
    pickle.dump(conf_mat_data, f)   

Producing results for different sampling techniques - working data only:   0%|          | 0/15 [00:00<?, ?it/s]

Analysing 1s_no


Producing results for different sampling techniques - working data only:   7%|▋         | 1/15 [00:11<02:47, 11.99s/it]

Analysing 1s_0.5


Producing results for different sampling techniques - working data only:  13%|█▎        | 2/15 [00:26<02:58, 13.75s/it]

...

Producing results for different sampling techniques - working data only:  87%|████████▋ | 13/15 [02:04<00:13,  6.82s/it]

Analysing 5s_0.5


Producing results for different sampling techniques - working data only:  93%|█████████▎| 14/15 [02:10<00:06,  6.46s/it]

Analysing 5s_0.8


Producing results for different sampling techniques - working data only: 100%|██████████| 15/15 [02:18<00:00,  9.26s/it]
